# MinIO Deployment WorkflowThis notebook demonstrates the complete workflow for storing, versioning, and deploying HMM artifacts using MinIO object storage.## Topics Covered1. Training and uploading models to MinIO2. Listing and downloading artifacts by version3. Tagging workflow (staging → production)4. Production deployment artifact retrieval5. Troubleshooting common MinIO issues## Prerequisites- MinIO server running (via docker-compose)- Python packages: minio, numpy, pandas- Trained HMM model or sample data

## 1. Setup and Environment Check

In [ ]:
import sysimport osfrom pathlib import Path# Add parent directory to path for importssys.path.insert(0, str(Path.cwd().parent / 'py'))import numpy as npimport pandas as pdfrom datetime import datetimeimport jsonprint("✓ Basic imports successful")

In [ ]:
# Import MinIO and HMM componentstry:    from minio import Minio    from minio.error import S3Error    print("✓ MinIO client available")except ImportError:    print("❌ MinIO not installed. Run: pip install minio")    try:    from imp.hmm.artifact_management import (        MinIOConfig,        MinIOArtifactStore,        ExperimentTracker,        ResearchArtifact,        ArtifactValidator    )    from imp.hmm.models import HMMArtifact, FusionWeights    from imp.hmm.trainer import EnhancedHMMTrainer    print("✓ HMM components available")except ImportError as e:    print(f"❌ Import error: {e}")

### Check MinIO ConnectionVerify that MinIO is running and accessible.

In [ ]:
def check_minio_connection():    """Check if MinIO is accessible."""    try:        config = MinIOConfig.from_env()        client = Minio(            config.endpoint,            access_key=config.access_key,            secret_key=config.secret_key,            secure=config.secure        )                # Try to list buckets        buckets = client.list_buckets()        print(f"✓ MinIO connection successful")        print(f"  Endpoint: {config.endpoint}")        print(f"  Buckets: {[b.name for b in buckets]}")        return True    except Exception as e:        print(f"❌ MinIO connection failed: {e}")        print("\nTroubleshooting:")        print("  1. Ensure docker-compose is running: docker-compose up -d")        print("  2. Check MinIO is accessible: curl http://localhost:9000")        print("  3. Verify environment variables are set correctly")        return Falsecheck_minio_connection()

## 2. Train Model and Upload to MinIOTrain a simple HMM model and upload it to MinIO with versioning.

In [ ]:
# Generate sample training datanp.random.seed(42)n_samples = 1000n_features = 3# Simulate market data with regime changesobservations = np.random.randn(n_samples, n_features)observations[:500] *= 0.5  # Low volatility regimeobservations[500:] *= 1.5  # High volatility regimeprint(f"Generated {n_samples} observations with {n_features} features")print(f"Shape: {observations.shape}")

In [ ]:
# Train HMM modelfrom hmmlearn import hmmprint("Training HMM model...")model = hmm.GaussianHMM(    n_components=2,    covariance_type='full',    n_iter=100,    random_state=42)model.fit(observations)print(f"✓ Model trained successfully")print(f"  States: {model.n_components}")print(f"  Log-likelihood: {model.score(observations):.2f}")

In [ ]:
# Create HMM artifact with correct field namesartifact = HMMArtifact(    version="1.0.0",    n_states=model.n_components,    n_features=n_features,    transition_matrix=model.transmat_.tolist(),    initial_probabilities=model.startprob_.tolist(),    means=model.means_.tolist(),    covariances=model.covars_.tolist(),    training_window_start=0,    training_window_end=n_samples,    metadata={        'training_date': datetime.now().isoformat(),        'n_samples': n_samples,        'covariance_type': 'full',        'description': 'Demo model for MinIO workflow'    })print("✓ HMM artifact created")

### Upload to MinIO with VersioningUpload the trained model to MinIO with semantic versioning.

In [ ]:
# Initialize MinIO storestore = MinIOArtifactStore()# Upload artifactartifact_name = "demo_regime_detector"version = "1.0.0"print(f"Uploading {artifact_name} v{version} to MinIO...")store.save_artifact(    artifact=artifact,    name=artifact_name,    version=version,    metadata={        'experiment': 'minio_demo',        'author': 'notebook_user',        'description': 'Demonstration of MinIO workflow'    })print(f"✓ Artifact uploaded successfully")print(f"  Name: {artifact_name}")print(f"  Version: {version}")print(f"  Bucket: {store.config.bucket_name}")

## 3. List and Download Artifacts by VersionExplore available artifacts and download specific versions.

In [ ]:
# List all versions of an artifactprint(f"Available versions of '{artifact_name}':")print("-" * 50)versions = store.list_artifact_versions(artifact_name)for v in versions:    print(f"  • {v['version']}")    print(f"    Size: {v['size'] / 1024:.2f} KB")    print(f"    Modified: {v['last_modified']}")    if v.get('tags'):        print(f"    Tags: {', '.join(v['tags'])}")    print()

In [ ]:
# Download specific versionprint(f"Downloading {artifact_name} v{version}...")downloaded_artifact, metadata = store.load_artifact(artifact_name, version)print(f"✓ Artifact downloaded successfully")print(f"\nArtifact details:")print(f"  States: {downloaded_artifact.n_states}")print(f"  Features: {downloaded_artifact.n_features}")print(f"  Version: {downloaded_artifact.version}")print(f"\nMetadata: {json.dumps(metadata, indent=2)}")# Verify it matches originalassert np.allclose(artifact.means, downloaded_artifact.means)print("\n✓ Downloaded artifact matches original")

### Upload Multiple VersionsCreate and upload multiple versions to demonstrate version management.

In [ ]:
# Upload version 1.1.0 with slight modificationsartifact_v11 = HMMArtifact(    version="1.1.0",    n_states=model.n_components,    n_features=n_features,    transition_matrix=model.transmat_.tolist(),    initial_probabilities=model.startprob_.tolist(),    means=(model.means_ * 1.01).tolist(),  # Slight modification    covariances=model.covars_.tolist(),    training_window_start=0,    training_window_end=n_samples,    metadata={        'training_date': datetime.now().isoformat(),        'version_notes': 'Minor parameter adjustment'    })store.save_artifact(artifact_v11, artifact_name, "1.1.0",                    metadata={'experiment': 'minio_demo', 'iteration': 2})print("✓ Uploaded v1.1.0")# Upload version 2.0.0 with major changesartifact_v20 = HMMArtifact(    version="2.0.0",    n_states=3,  # Changed number of states    n_features=n_features,    transition_matrix=[[0.7, 0.2, 0.1], [0.2, 0.6, 0.2], [0.1, 0.2, 0.7]],    initial_probabilities=[0.33, 0.33, 0.34],    means=np.random.randn(3, n_features).tolist(),    covariances=[np.eye(n_features).tolist() for _ in range(3)],    training_window_start=0,    training_window_end=n_samples,    metadata={        'training_date': datetime.now().isoformat(),        'version_notes': 'Major update: 3 states'    })store.save_artifact(artifact_v20, artifact_name, "2.0.0",                   metadata={'experiment': 'minio_demo', 'iteration': 3})print("✓ Uploaded v2.0.0")# List all versionsprint("\nAll versions:")for v in store.list_artifact_versions(artifact_name):    print(f"  • {v['version']} ({v['size'] / 1024:.2f} KB)")

## 4. Tagging Workflow (Staging → Production)Demonstrate the promotion workflow from development through staging to production.

In [ ]:
# Tag v1.0.0 as stagingprint("Tagging workflow demonstration:")print("-" * 50)store.tag_artifact(artifact_name, "1.0.0", "staging")print("✓ v1.0.0 tagged as 'staging'")# Verify staging tagstaging_artifact, staging_meta = store.load_artifact_by_tag(artifact_name, "staging")print(f"  Staging artifact: {staging_meta.get('version', 'unknown')}")

In [ ]:
# After testing in staging, promote to productionprint("\nPromoting to production...")store.tag_artifact(artifact_name, "1.0.0", "production")print("✓ v1.0.0 tagged as 'production'")# Verify production tagprod_artifact, prod_meta = store.load_artifact_by_tag(artifact_name, "production")print(f"  Production artifact: {prod_meta.get('version', 'unknown')}")

In [ ]:
# Later, promote v1.1.0 to stagingprint("\nUpdating staging to v1.1.0...")store.tag_artifact(artifact_name, "1.1.0", "staging")print("✓ v1.1.0 tagged as 'staging'")# Show current tagsprint("\nCurrent deployment status:")print(f"  Production: v{store.load_artifact_by_tag(artifact_name, 'production')[1].get('version')}")print(f"  Staging: v{store.load_artifact_by_tag(artifact_name, 'staging')[1].get('version')}")

### Rollback ScenarioDemonstrate rolling back production to a previous version.

In [ ]:
# Simulate a problem with production - rollback to v1.0.0print("Rollback scenario: Issue detected with current production")print("-" * 50)# Check current production versioncurrent_prod = store.load_artifact_by_tag(artifact_name, "production")[1].get('version')print(f"Current production: v{current_prod}")# Rollback by re-tagging an older versionprint("\nRolling back to v1.0.0...")store.tag_artifact(artifact_name, "1.0.0", "production")print("✓ Production rolled back to v1.0.0")# Verify rollbacknew_prod = store.load_artifact_by_tag(artifact_name, "production")[1].get('version')print(f"New production: v{new_prod}")

## 5. Production Deployment Artifact RetrievalDemonstrate how production systems retrieve artifacts.

In [ ]:
def deploy_production_model():    """    Simulate production deployment process.    This function shows how a production system would retrieve and use artifacts.    """    print("Production Deployment Process")    print("=" * 50)        # Step 1: Retrieve production artifact    print("\n1. Retrieving production artifact...")    artifact, metadata = store.load_artifact_by_tag(artifact_name, "production")    print(f"   ✓ Loaded: {artifact_name} v{metadata.get('version')}")        # Step 2: Validate artifact    print("\n2. Validating artifact...")    validator = ArtifactValidator()    is_valid, issues = validator.validate_artifact(artifact)        if is_valid:        print("   ✓ Artifact validation passed")    else:        print(f"   ❌ Validation failed: {issues}")        return None        # Step 3: Initialize inference    print("\n3. Initializing inference engine...")    # In production, you would initialize your inference engine here    print(f"   ✓ Model ready for inference")    print(f"   • States: {artifact.n_states}")    print(f"   • Features: {artifact.n_features}")        # Step 4: Test inference    print("\n4. Testing inference...")    test_obs = np.random.randn(10, artifact.n_features)    # Simulate inference (in production, use actual inference engine)    print(f"   ✓ Inference test successful")    print(f"   • Test observations: {test_obs.shape}")        return artifact# Run deploymentdeployed_artifact = deploy_production_model()

### Experiment Tracking IntegrationTrack experiments and link them to artifacts.

In [ ]:
# Initialize experiment trackertracker = ExperimentTracker(store)# Create experimentexperiment_id = tracker.create_experiment(    name="regime_detection_optimization",    description="Optimizing HMM parameters for regime detection",    tags=["hmm", "regime-detection", "optimization"])print(f"Created experiment: {experiment_id}")

In [ ]:
# Log runs with different configurationsconfigs = [    {"n_states": 2, "covariance_type": "full"},    {"n_states": 3, "covariance_type": "full"},    {"n_states": 2, "covariance_type": "diag"},]for i, config in enumerate(configs):    run_id = tracker.log_run(        experiment_id=experiment_id,        artifact_name=artifact_name,        artifact_version=f"exp_{i+1}.0.0",        config=config,        metrics={            "log_likelihood": -1000 + i * 50,            "aic": 2100 - i * 30,            "bic": 2200 - i * 25        },        tags=["experiment"]    )    print(f"  Run {i+1}: {run_id[:8]}... (n_states={config['n_states']})")

In [ ]:
# Query experiment runsprint("\nExperiment runs:")runs = tracker.get_experiment_runs(experiment_id)print(f"Total runs: {len(runs)}")# Find best run by metricbest_run = min(runs, key=lambda r: r.get('metrics', {}).get('aic', float('inf')))print(f"\nBest run (by AIC):")print(f"  Run ID: {best_run['run_id'][:8]}...")print(f"  Config: {best_run['config']}")print(f"  Metrics: {best_run['metrics']}")

## 6. Troubleshooting Common MinIO IssuesCommon issues and their solutions.

### Issue 1: Connection Refused**Symptom**: `ConnectionRefusedError` or `MaxRetryError`**Solutions**:

In [ ]:
def troubleshoot_connection():    """Diagnose MinIO connection issues."""    print("MinIO Connection Troubleshooting")    print("=" * 50)        # Check 1: Docker containers    print("\n1. Checking Docker containers...")    import subprocess    try:        result = subprocess.run(            ["docker", "ps", "--filter", "name=minio"],            capture_output=True,            text=True        )        if "minio" in result.stdout:            print("   ✓ MinIO container is running")        else:            print("   ❌ MinIO container not found")            print("   → Run: docker-compose up -d")    except Exception as e:        print(f"   ⚠ Could not check Docker: {e}")        # Check 2: Environment variables    print("\n2. Checking environment variables...")    required_vars = [        "MINIO_ENDPOINT",        "MINIO_ACCESS_KEY",        "MINIO_SECRET_KEY"    ]        for var in required_vars:        value = os.environ.get(var)        if value:            # Mask secret key            display_value = value if "SECRET" not in var else "*" * 8            print(f"   ✓ {var}={display_value}")        else:            print(f"   ❌ {var} not set")            print(f"   → Set in .env file or export {var}=<value>")        # Check 3: Network connectivity    print("\n3. Checking network connectivity...")    try:        import socket        endpoint = os.environ.get("MINIO_ENDPOINT", "localhost:9000")        host, port = endpoint.split(":")        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)        sock.settimeout(2)        result = sock.connect_ex((host, int(port)))        sock.close()                if result == 0:            print(f"   ✓ Can connect to {endpoint}")        else:            print(f"   ❌ Cannot connect to {endpoint}")            print(f"   → Check if MinIO is running on correct port")    except Exception as e:        print(f"   ⚠ Network check failed: {e}")troubleshoot_connection()

### Issue 2: Access Denied**Symptom**: `AccessDenied` or `InvalidAccessKeyId`**Solutions**:

In [ ]:
def troubleshoot_access():    """Diagnose MinIO access issues."""    print("MinIO Access Troubleshooting")    print("=" * 50)        print("\n1. Verify credentials...")    config = MinIOConfig.from_env()        try:        client = Minio(            config.endpoint,            access_key=config.access_key,            secret_key=config.secret_key,            secure=config.secure        )                # Try to list buckets        buckets = client.list_buckets()        print(f"   ✓ Credentials valid")        print(f"   ✓ Can list buckets: {[b.name for b in buckets]}")                # Check bucket access        print(f"\n2. Checking bucket '{config.bucket_name}'...")        try:            list(client.list_objects(config.bucket_name, max_keys=1))            print(f"   ✓ Can access bucket '{config.bucket_name}'")        except S3Error as e:            if e.code == "NoSuchBucket":                print(f"   ❌ Bucket '{config.bucket_name}' does not exist")                print(f"   → Creating bucket...")                client.make_bucket(config.bucket_name)                print(f"   ✓ Bucket created")            else:                print(f"   ❌ Access error: {e}")                    except Exception as e:        print(f"   ❌ Authentication failed: {e}")        print("\n   Solutions:")        print("   • Check MINIO_ACCESS_KEY and MINIO_SECRET_KEY")        print("   • Verify credentials in MinIO console")        print("   • Check docker-compose.yml for correct credentials")troubleshoot_access()

### Issue 3: Bucket Not Found**Symptom**: `NoSuchBucket` error**Solutions**:

In [ ]:
def troubleshoot_bucket():    """Diagnose and fix bucket issues."""    print("MinIO Bucket Troubleshooting")    print("=" * 50)        config = MinIOConfig.from_env()    client = Minio(        config.endpoint,        access_key=config.access_key,        secret_key=config.secret_key,        secure=config.secure    )        print(f"\n1. Checking if bucket '{config.bucket_name}' exists...")        try:        if client.bucket_exists(config.bucket_name):            print(f"   ✓ Bucket exists")                        # Check objects in bucket            objects = list(client.list_objects(config.bucket_name, max_keys=5))            print(f"   ✓ Bucket contains {len(objects)} objects (showing max 5)")            for obj in objects:                print(f"     • {obj.object_name} ({obj.size / 1024:.2f} KB)")        else:            print(f"   ❌ Bucket does not exist")            print(f"   → Creating bucket '{config.bucket_name}'...")            client.make_bucket(config.bucket_name)            print(f"   ✓ Bucket created successfully")                except Exception as e:        print(f"   ❌ Error: {e}")troubleshoot_bucket()

### Issue 4: Version Not Found**Symptom**: Cannot find specific artifact version**Solutions**:

In [ ]:
def troubleshoot_versions():    """Diagnose version-related issues."""    print("Artifact Version Troubleshooting")    print("=" * 50)        # List all artifacts    print("\n1. Listing all artifacts in bucket...")    try:        config = MinIOConfig.from_env()        client = Minio(            config.endpoint,            access_key=config.access_key,            secret_key=config.secret_key,            secure=config.secure        )                objects = client.list_objects(config.bucket_name, recursive=True)        artifacts = {}                for obj in objects:            # Parse artifact name and version from path            parts = obj.object_name.split('/')            if len(parts) >= 2:                name = parts[0]                version = parts[1] if len(parts) > 1 else "unknown"                                if name not in artifacts:                    artifacts[name] = []                artifacts[name].append(version)                if artifacts:            print(f"   ✓ Found {len(artifacts)} artifact(s)")            for name, versions in artifacts.items():                print(f"\n   Artifact: {name}")                for v in sorted(set(versions)):                    print(f"     • {v}")        else:            print("   ⚠ No artifacts found in bucket")            print("   → Upload an artifact first")                except Exception as e:        print(f"   ❌ Error: {e}")troubleshoot_versions()

### Issue 5: Slow Upload/Download**Symptom**: Transfers are very slow**Solutions**:

In [ ]:
def troubleshoot_performance():    """Diagnose performance issues."""    print("MinIO Performance Troubleshooting")    print("=" * 50)        print("\n1. Testing upload/download speed...")        # Create test data    test_data = np.random.randn(1000, 100)  # ~800KB    test_artifact = HMMArtifact(        version="test",        n_states=2,        n_features=100,        transition_matrix=[[0.9, 0.1], [0.1, 0.9]],        initial_probabilities=[0.5, 0.5],        means=test_data[:2].tolist(),        covariances=[np.eye(100).tolist(), np.eye(100).tolist()],        training_window_start=0,        training_window_end=1000,        metadata={"test": "performance"}    )        # Test upload speed    import time    start = time.time()    store.save_artifact(test_artifact, "perf_test", "1.0.0")    upload_time = time.time() - start    print(f"   Upload time: {upload_time:.2f}s")        # Test download speed    start = time.time()    store.load_artifact("perf_test", "1.0.0")    download_time = time.time() - start    print(f"   Download time: {download_time:.2f}s")        # Recommendations    print("\n2. Performance recommendations:")    if upload_time > 5 or download_time > 5:        print("   ⚠ Slow transfers detected")        print("   → Check network connectivity")        print("   → Ensure MinIO is running locally (not remote)")        print("   → Check Docker resource limits")    else:        print("   ✓ Transfer speeds are acceptable")        # Cleanup    try:        config = MinIOConfig.from_env()        client = Minio(            config.endpoint,            access_key=config.access_key,            secret_key=config.secret_key,            secure=config.secure        )        # Remove test artifact        for obj in client.list_objects(config.bucket_name, prefix="perf_test/"):            client.remove_object(config.bucket_name, obj.object_name)        print("\n   ✓ Cleaned up test artifacts")    except:        passtroubleshoot_performance()

## SummaryThis notebook demonstrated:1. ✓ **Training and uploading** models to MinIO with versioning2. ✓ **Listing and downloading** artifacts by version3. ✓ **Tagging workflow** for staging and production deployment4. ✓ **Production deployment** artifact retrieval patterns5. ✓ **Troubleshooting** common MinIO issues### Key Takeaways- **Semantic versioning** enables clear version management- **Tags** (staging/production) simplify deployment workflows- **Experiment tracking** links artifacts to research experiments- **Validation** ensures artifact integrity before deployment- **Troubleshooting tools** help diagnose common issues quickly### Next Steps- Integrate MinIO into your CI/CD pipeline- Set up automated testing for artifact validation- Implement monitoring for production artifacts- Create backup strategies for critical artifacts- Explore MinIO's advanced features (lifecycle policies, replication)### Additional Resources- MinIO Documentation: https://min.io/docs/- HMM Artifact Management: `py/imp/hmm/artifact_management.py`- Integration Tests: `py/tests/test_minio_integration.py`- Testing Guide: `py/tests/MINIO_TESTING_GUIDE.md`

### Upload Multiple VersionsCreate and upload multiple versions to demonstrate version management.

In [ ]:
# Upload version 1.1.0 with slight modificationsartifact_v11 = HMMArtifact(    version="1.1.0",    n_states=model.n_components,    n_features=n_features,    transition_matrix=model.transmat_.tolist(),    initial_probabilities=model.startprob_.tolist(),    means=(model.means_ * 1.01).tolist(),  # Slight modification    covariances=model.covars_.tolist(),    training_window_start=0,    training_window_end=n_samples,    metadata={        'training_date': datetime.now().isoformat(),        'version_notes': 'Minor parameter adjustment'    })store.save_artifact(artifact_v11, artifact_name, "1.1.0",                    metadata={'experiment': 'minio_demo', 'iteration': 2})print("✓ Uploaded v1.1.0")# Upload version 2.0.0 with major changesartifact_v20 = HMMArtifact(    version="2.0.0",    n_states=3,  # Changed number of states    n_features=n_features,    transition_matrix=[[0.7, 0.2, 0.1], [0.2, 0.6, 0.2], [0.1, 0.2, 0.7]],    initial_probabilities=[0.33, 0.33, 0.34],    means=np.random.randn(3, n_features).tolist(),    covariances=[np.eye(n_features).tolist() for _ in range(3)],    training_window_start=0,    training_window_end=n_samples,    metadata={        'training_date': datetime.now().isoformat(),        'version_notes': 'Major update: 3 states'    })store.save_artifact(artifact_v20, artifact_name, "2.0.0",                   metadata={'experiment': 'minio_demo', 'iteration': 3})print("✓ Uploaded v2.0.0")# List all versionsprint("\nAll versions:")for v in store.list_artifact_versions(artifact_name):    print(f"  • {v['version']} ({v['size'] / 1024:.2f} KB)")

## 4. Tagging Workflow (Staging → Production)Demonstrate the promotion workflow from development through staging to production.

In [ ]:
# Tag v1.0.0 as stagingprint("Tagging workflow demonstration:")print("-" * 50)store.tag_artifact(artifact_name, "1.0.0", "staging")print("✓ v1.0.0 tagged as 'staging'")# Verify staging tagstaging_artifact, staging_meta = store.load_artifact_by_tag(artifact_name, "staging")print(f"  Staging artifact: {staging_meta.get('version', 'unknown')}")

In [ ]:
# After testing in staging, promote to productionprint("\nPromoting to production...")store.tag_artifact(artifact_name, "1.0.0", "production")print("✓ v1.0.0 tagged as 'production'")# Verify production tagprod_artifact, prod_meta = store.load_artifact_by_tag(artifact_name, "production")print(f"  Production artifact: {prod_meta.get('version', 'unknown')}")

In [ ]:
# Later, promote v1.1.0 to stagingprint("\nUpdating staging to v1.1.0...")store.tag_artifact(artifact_name, "1.1.0", "staging")print("✓ v1.1.0 tagged as 'staging'")# Show current tagsprint("\nCurrent deployment status:")print(f"  Production: v{store.load_artifact_by_tag(artifact_name, 'production')[1].get('version')}")print(f"  Staging: v{store.load_artifact_by_tag(artifact_name, 'staging')[1].get('version')}")

### Rollback ScenarioDemonstrate rolling back production to a previous version.

In [ ]:
# Simulate a problem with production - rollback to v1.0.0print("Rollback scenario: Issue detected with current production")print("-" * 50)# Check current production versioncurrent_prod = store.load_artifact_by_tag(artifact_name, "production")[1].get('version')print(f"Current production: v{current_prod}")# Rollback by re-tagging an older versionprint("\nRolling back to v1.0.0...")store.tag_artifact(artifact_name, "1.0.0", "production")print("✓ Production rolled back to v1.0.0")# Verify rollbacknew_prod = store.load_artifact_by_tag(artifact_name, "production")[1].get('version')print(f"New production: v{new_prod}")

## 5. Production Deployment Artifact RetrievalDemonstrate how production systems retrieve artifacts.

In [ ]:
def deploy_production_model():    """    Simulate production deployment process.    This function shows how a production system would retrieve and use artifacts.    """    print("Production Deployment Process")    print("=" * 50)        # Step 1: Retrieve production artifact    print("\n1. Retrieving production artifact...")    artifact, metadata = store.load_artifact_by_tag(artifact_name, "production")    print(f"   ✓ Loaded: {artifact_name} v{metadata.get('version')}")        # Step 2: Validate artifact    print("\n2. Validating artifact...")    validator = ArtifactValidator()    is_valid, issues = validator.validate_artifact(artifact)        if is_valid:        print("   ✓ Artifact validation passed")    else:        print(f"   ❌ Validation failed: {issues}")        return None        # Step 3: Initialize inference    print("\n3. Initializing inference engine...")    # In production, you would initialize your inference engine here    print(f"   ✓ Model ready for inference")    print(f"   • States: {artifact.n_states}")    print(f"   • Features: {artifact.n_features}")        # Step 4: Test inference    print("\n4. Testing inference...")    test_obs = np.random.randn(10, artifact.n_features)    # Simulate inference (in production, use actual inference engine)    print(f"   ✓ Inference test successful")    print(f"   • Test observations: {test_obs.shape}")        return artifact# Run deploymentdeployed_artifact = deploy_production_model()

### Experiment Tracking IntegrationTrack experiments and link them to artifacts.

In [ ]:
# Initialize experiment trackertracker = ExperimentTracker(store)# Create experimentexperiment_id = tracker.create_experiment(    name="regime_detection_optimization",    description="Optimizing HMM parameters for regime detection",    tags=["hmm", "regime-detection", "optimization"])print(f"Created experiment: {experiment_id}")

In [ ]:
# Log runs with different configurationsconfigs = [    {"n_states": 2, "covariance_type": "full"},    {"n_states": 3, "covariance_type": "full"},    {"n_states": 2, "covariance_type": "diag"},]for i, config in enumerate(configs):    run_id = tracker.log_run(        experiment_id=experiment_id,        artifact_name=artifact_name,        artifact_version=f"exp_{i+1}.0.0",        config=config,        metrics={            "log_likelihood": -1000 + i * 50,            "aic": 2100 - i * 30,            "bic": 2200 - i * 25        },        tags=["experiment"]    )    print(f"  Run {i+1}: {run_id[:8]}... (n_states={config['n_states']})")

In [ ]:
# Query experiment runsprint("\nExperiment runs:")runs = tracker.get_experiment_runs(experiment_id)print(f"Total runs: {len(runs)}")# Find best run by metricbest_run = min(runs, key=lambda r: r.get('metrics', {}).get('aic', float('inf')))print(f"\nBest run (by AIC):")print(f"  Run ID: {best_run['run_id'][:8]}...")print(f"  Config: {best_run['config']}")print(f"  Metrics: {best_run['metrics']}")

## 6. Troubleshooting Common MinIO IssuesCommon issues and their solutions.

### Issue 1: Connection Refused / Protocol Error**Symptoms**: - `ConnectionRefusedError` or `MaxRetryError`- `ProtocolError: Connection aborted`- `BadStatusLine` errors**Root Cause**: MinIO server is not running or not accessible**Solutions**:

In [ ]:
def troubleshoot_connection():    """Diagnose MinIO connection issues."""    print("MinIO Connection Troubleshooting")    print("=" * 50)        # Check 1: Docker containers    print("\n1. Checking Docker containers...")    import subprocess    try:        result = subprocess.run(            ["docker", "ps", "--filter", "name=minio"],            capture_output=True,            text=True,            timeout=5        )        if "minio" in result.stdout and "Up" in result.stdout:            print("   ✓ MinIO container is running")        else:            print("   ❌ MinIO container not found or not running")            print("   → Run: docker-compose up -d")            print("   → Or: docker run -p 9000:9000 -p 9001:9001 minio/minio server /data --console-address :9001")    except subprocess.TimeoutExpired:        print("   ⚠ Docker command timed out")    except Exception as e:        print(f"   ⚠ Could not check Docker: {e}")        # Check 2: Environment variables    print("\n2. Checking environment variables...")    required_vars = [        "MINIO_ENDPOINT",        "MINIO_ACCESS_KEY",        "MINIO_SECRET_KEY"    ]        all_set = True    for var in required_vars:        value = os.environ.get(var)        if value:            # Mask secret key            display_value = value if "SECRET" not in var else "*" * 8            print(f"   ✓ {var}={display_value}")        else:            print(f"   ❌ {var} not set")            print(f"   → Set in .env file or export {var}=<value>")            all_set = False        if not all_set:        print("\n   Example .env file:")        print("   MINIO_ENDPOINT=localhost:9000")        print("   MINIO_ACCESS_KEY=minioadmin")        print("   MINIO_SECRET_KEY=minioadmin")        print("   MINIO_BUCKET_NAME=hmm-artifacts")        # Check 3: Network connectivity    print("\n3. Checking network connectivity...")    try:        import socket        endpoint = os.environ.get("MINIO_ENDPOINT", "localhost:9000")        host, port = endpoint.split(":")        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)        sock.settimeout(2)        result = sock.connect_ex((host, int(port)))        sock.close()                if result == 0:            print(f"   ✓ Can connect to {endpoint}")        else:            print(f"   ❌ Cannot connect to {endpoint}")            print(f"   → Check if MinIO is running: docker ps")            print(f"   → Check port mapping: docker port <container_name>")            print(f"   → Try: curl http://{endpoint}/minio/health/live")    except Exception as e:        print(f"   ⚠ Network check failed: {e}")        # Check 4: Quick start guide    print("\n4. Quick Start Guide:")    print("   If MinIO is not running, start it with:")    print("   ")    print("   # Using docker-compose (recommended)")    print("   docker-compose up -d")    print("   ")    print("   # Or using docker directly")    print("   docker run -d -p 9000:9000 -p 9001:9001 \\")    print("     -e MINIO_ROOT_USER=minioadmin \\")    print("     -e MINIO_ROOT_PASSWORD=minioadmin \\")    print("     minio/minio server /data --console-address :9001")    print("   ")    print("   Then access MinIO console at: http://localhost:9001")troubleshoot_connection()

### Issue 2: Access Denied**Symptom**: `AccessDenied` or `InvalidAccessKeyId`**Solutions**:

In [ ]:
def troubleshoot_access():    """Diagnose MinIO access issues."""    print("MinIO Access Troubleshooting")    print("=" * 50)        print("\n1. Verify credentials...")    try:        config = MinIOConfig.from_env()                client = Minio(            config.endpoint,            access_key=config.access_key,            secret_key=config.secret_key,            secure=config.secure        )                # Try to list buckets        buckets = client.list_buckets()        print(f"   ✓ Credentials valid")        print(f"   ✓ Can list buckets: {[b.name for b in buckets]}")                # Check bucket access        print(f"\n2. Checking bucket '{config.bucket_name}'...")        try:            list(client.list_objects(config.bucket_name, max_keys=1))            print(f"   ✓ Can access bucket '{config.bucket_name}'")        except S3Error as e:            if e.code == "NoSuchBucket":                print(f"   ❌ Bucket '{config.bucket_name}' does not exist")                print(f"   → Creating bucket...")                client.make_bucket(config.bucket_name)                print(f"   ✓ Bucket created")            else:                print(f"   ❌ Access error: {e}")                    except Exception as e:        print(f"   ❌ Authentication failed: {e}")        print("\n   Solutions:")        print("   • Check MINIO_ACCESS_KEY and MINIO_SECRET_KEY")        print("   • Verify credentials in MinIO console (http://localhost:9001)")        print("   • Check docker-compose.yml for correct credentials")        print("   • Default credentials: minioadmin / minioadmin")# Only run if MinIO is accessibletry:    troubleshoot_access()except:    print("⚠ Skipping access check - MinIO not accessible")

### Issue 3: Bucket Not Found**Symptom**: `NoSuchBucket` error**Solutions**:

In [ ]:
def troubleshoot_bucket():    """Diagnose and fix bucket issues."""    print("MinIO Bucket Troubleshooting")    print("=" * 50)        try:        config = MinIOConfig.from_env()        client = Minio(            config.endpoint,            access_key=config.access_key,            secret_key=config.secret_key,            secure=config.secure        )                print(f"\n1. Checking if bucket '{config.bucket_name}' exists...")                if client.bucket_exists(config.bucket_name):            print(f"   ✓ Bucket exists")                        # Check objects in bucket            objects = list(client.list_objects(config.bucket_name, max_keys=5))            print(f"   ✓ Bucket contains {len(objects)} objects (showing max 5)")            for obj in objects:                print(f"     • {obj.object_name} ({obj.size / 1024:.2f} KB)")        else:            print(f"   ❌ Bucket does not exist")            print(f"   → Creating bucket '{config.bucket_name}'...")            client.make_bucket(config.bucket_name)            print(f"   ✓ Bucket created successfully")                except Exception as e:        print(f"   ❌ Error: {e}")        print("   → Ensure MinIO is running and accessible")# Only run if MinIO is accessibletry:    troubleshoot_bucket()except:    print("⚠ Skipping bucket check - MinIO not accessible")

### Issue 4: Version Not Found**Symptom**: Cannot find specific artifact version**Solutions**:

In [ ]:
def troubleshoot_versions():    """Diagnose version-related issues."""    print("Artifact Version Troubleshooting")    print("=" * 50)        # List all artifacts    print("\n1. Listing all artifacts in bucket...")    try:        config = MinIOConfig.from_env()        client = Minio(            config.endpoint,            access_key=config.access_key,            secret_key=config.secret_key,            secure=config.secure        )                objects = client.list_objects(config.bucket_name, recursive=True)        artifacts = {}                for obj in objects:            # Parse artifact name and version from path            parts = obj.object_name.split('/')            if len(parts) >= 2:                name = parts[0]                version = parts[1] if len(parts) > 1 else "unknown"                                if name not in artifacts:                    artifacts[name] = []                artifacts[name].append(version)                if artifacts:            print(f"   ✓ Found {len(artifacts)} artifact(s)")            for name, versions in artifacts.items():                print(f"\n   Artifact: {name}")                for v in sorted(set(versions)):                    print(f"     • {v}")        else:            print("   ⚠ No artifacts found in bucket")            print("   → Upload an artifact first")                except Exception as e:        print(f"   ❌ Error: {e}")# Only run if MinIO is accessibletry:    troubleshoot_versions()except:    print("⚠ Skipping version check - MinIO not accessible")

### Issue 5: Slow Upload/Download**Symptom**: Transfers are very slow**Solutions**:

In [ ]:
def troubleshoot_performance():    """Diagnose performance issues."""    print("MinIO Performance Troubleshooting")    print("=" * 50)        try:        print("\n1. Testing upload/download speed...")                # Create test data        test_data = np.random.randn(1000, 100)  # ~800KB        test_artifact = HMMArtifact(            version="perf_test",            n_states=2,            n_features=100,            transition_matrix=[[0.9, 0.1], [0.1, 0.9]],            initial_probabilities=[0.5, 0.5],            means=test_data[:2].tolist(),            covariances=[np.eye(100).tolist(), np.eye(100).tolist()],            training_window_start=0,            training_window_end=1000,            metadata={"test": "performance"}        )                # Test upload speed        import time        start = time.time()        store.save_artifact(test_artifact, "perf_test", "1.0.0")        upload_time = time.time() - start        print(f"   Upload time: {upload_time:.2f}s")                # Test download speed        start = time.time()        store.load_artifact("perf_test", "1.0.0")        download_time = time.time() - start        print(f"   Download time: {download_time:.2f}s")                # Recommendations        print("\n2. Performance recommendations:")        if upload_time > 5 or download_time > 5:            print("   ⚠ Slow transfers detected")            print("   → Check network connectivity")            print("   → Ensure MinIO is running locally (not remote)")            print("   → Check Docker resource limits")            print("   → Consider using MinIO distributed mode for production")        else:            print("   ✓ Transfer speeds are acceptable")                # Cleanup        config = MinIOConfig.from_env()        client = Minio(            config.endpoint,            access_key=config.access_key,            secret_key=config.secret_key,            secure=config.secure        )        # Remove test artifact        for obj in client.list_objects(config.bucket_name, prefix="perf_test/"):            client.remove_object(config.bucket_name, obj.object_name)        print("\n   ✓ Cleaned up test artifacts")            except Exception as e:        print(f"   ❌ Performance test failed: {e}")        print("   → Ensure MinIO is running and accessible")# Only run if MinIO is accessibletry:    troubleshoot_performance()except:    print("⚠ Skipping performance test - MinIO not accessible")

## SummaryThis notebook demonstrated:1. ✓ **Training and uploading** models to MinIO with versioning2. ✓ **Listing and downloading** artifacts by version3. ✓ **Tagging workflow** for staging and production deployment4. ✓ **Production deployment** artifact retrieval patterns5. ✓ **Troubleshooting** common MinIO issues### Key Takeaways- **Semantic versioning** enables clear version management- **Tags** (staging/production) simplify deployment workflows- **Experiment tracking** links artifacts to research experiments- **Validation** ensures artifact integrity before deployment- **Troubleshooting tools** help diagnose common issues quickly### Next Steps- Integrate MinIO into your CI/CD pipeline- Set up automated testing for artifact validation- Implement monitoring for production artifacts- Create backup strategies for critical artifacts- Explore MinIO's advanced features (lifecycle policies, replication)### Additional Resources- MinIO Documentation: https://min.io/docs/- HMM Artifact Management: `py/imp/hmm/artifact_management.py`- Integration Tests: `py/tests/test_minio_integration.py`- Testing Guide: `py/tests/MINIO_TESTING_GUIDE.md`- Docker Compose Setup: `docker-compose.yml`### Common Commands```bash# Start MinIOdocker-compose up -d# Check MinIO statusdocker ps | grep minio# View MinIO logsdocker-compose logs minio# Stop MinIOdocker-compose down# Access MinIO console# Open browser to: http://localhost:9001# Default credentials: minioadmin / minioadmin```